# Task 3: Data Visualization — Movie Analytics

CodeAlpha Data Analytics Internship

Turning the findings from Task 2's EDA into a small set of polished, presentation-ready
charts. Each one is saved as a PNG into `visuals/` so you can drop them straight into
your LinkedIn post, blog write-up, or the video walkthrough.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

os.makedirs("visuals", exist_ok=True)

# A consistent, clean look across every chart in this notebook
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.6,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.dpi": 110,
})

PRIMARY = "#2E5EAA"
ACCENT = "#E4572E"
PALETTE = ["#2E5EAA", "#4C9F70", "#E4572E", "#F3A712", "#8E44AD", "#17A398", "#C0392B", "#5D6D7E"]


In [ ]:
df = pd.read_csv("data/movies_clean.csv")
df["genre_list"] = df["genre"].apply(lambda g: [x.strip() for x in str(g).split(",")] if pd.notna(g) else [])
print(f"{len(df)} movies loaded.")
df.head(3)


## 1. Which genres dominate the highest-grossing list?


In [ ]:
genre_counts = df.explode("genre_list")["genre_list"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(genre_counts.index, genre_counts.values, color=PRIMARY)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title("Most Common Genres Among the Highest-Grossing Films of All Time")
ax.set_xlabel("Number of films in the top-grossing list")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/genre_popularity.png", bbox_inches="tight")
plt.show()


## 2. Has the dominant genre shifted over time?


In [ ]:
top_genres = df.explode("genre_list")["genre_list"].value_counts().head(5).index.tolist()

by_decade = (
    df.explode("genre_list")
    .dropna(subset=["decade"])
    .query("genre_list in @top_genres")
    .groupby(["decade", "genre_list"])
    .size()
    .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(9, 5.5))
by_decade.plot(kind="bar", stacked=False, ax=ax, color=PALETTE[:len(top_genres)], width=0.75)
ax.set_title("Top 5 Genres by Decade")
ax.set_xlabel("Decade")
ax.set_ylabel("Number of films")
ax.legend(title="Genre", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("visuals/genre_by_decade.png", bbox_inches="tight")
plt.show()


## 3. Does IMDb rating predict box office success?


In [ ]:
plot_df = df.dropna(subset=["gross_usd", "imdb_rating"]).copy()
plot_df["gross_billion"] = plot_df["gross_usd"] / 1e9

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(plot_df["imdb_rating"], plot_df["gross_billion"], s=55, alpha=0.65, color=PRIMARY, edgecolor="white")

# Label the handful of standout points so the chart tells a story, not just dots
standout = pd.concat([
    plot_df.nlargest(3, "gross_billion"),
    plot_df.nsmallest(2, "imdb_rating"),
])
for _, row in standout.iterrows():
    ax.annotate(row["title"], (row["imdb_rating"], row["gross_billion"]),
                textcoords="offset points", xytext=(6, 6), fontsize=8.5, color="#333333")

corr = plot_df["imdb_rating"].corr(plot_df["gross_usd"])
ax.set_title(f"IMDb Rating vs. Worldwide Gross (correlation: {corr:.2f})")
ax.set_xlabel("IMDb Rating")
ax.set_ylabel("Worldwide Gross ($ Billions)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/rating_vs_gross.png", bbox_inches="tight")
plt.show()

print(f"Correlation coefficient: {corr:.2f} - " +
      ("a weak relationship." if abs(corr) < 0.3 else "a moderate relationship." if abs(corr) < 0.6 else "a strong relationship."))


## 4. Directors with the strongest average ratings

(Only directors with at least 2 films on this list, so a single outlier doesn't
dominate the ranking.)


In [ ]:
director_stats = (
    df.dropna(subset=["director", "imdb_rating"])
    .groupby("director")
    .agg(avg_rating=("imdb_rating", "mean"), films=("title", "count"))
    .query("films >= 2")
    .sort_values("avg_rating", ascending=False)
    .head(10)
    .sort_values("avg_rating")
)

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(director_stats.index, director_stats["avg_rating"], color=ACCENT)
ax.bar_label(bars, fmt="%.1f", padding=4, fontsize=10)
ax.set_title("Highest Average-Rated Directors (2+ films on the list)")
ax.set_xlabel("Average IMDb Rating")
ax.set_xlim(0, 10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/top_directors.png", bbox_inches="tight")
plt.show()


## 5. The top 10, for context


In [ ]:
top10 = df.dropna(subset=["gross_usd"]).nlargest(10, "gross_usd").sort_values("gross_usd")

fig, ax = plt.subplots(figsize=(8, 5.5))
bars = ax.barh(top10["title"], top10["gross_usd"] / 1e9, color=PRIMARY)
ax.bar_label(bars, fmt="$%.2fB", padding=4, fontsize=9.5)
ax.set_title("Top 10 Highest-Grossing Films of All Time")
ax.set_xlabel("Worldwide Gross ($ Billions)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/top10_grossing.png", bbox_inches="tight")
plt.show()


## Wrap-up

All five charts are saved in the `visuals/` folder as PNGs, ready to drop straight into
your LinkedIn post, blog write-up, or the internship video. Grab `rating_vs_gross.png`
and `genre_by_decade.png` first — those two carry the most interesting findings from
this project so far.

Next: **Task 4 — Sentiment Analysis** on real audience reviews, closing the loop on
whether the crowd's sentiment lines up with these critic ratings.
